In [1]:
from data import StorageManager
import pandas as pd
storage_man = StorageManager()

In [7]:
storage_man.put_object_restaurant_reviews()

In [2]:
df = storage_man.get_object('European Restaurant Reviews.csv')

TypeError: StorageManager.get_object() missing 2 required positional arguments: 'skiprows' and 'nrows'

In [3]:
df

,Country,Restaurant Name,Sentiment,Review Title,Review Date,Review
0,France,The Frog at Bercy Village,Negative,Rude manager,May 2024 •,The manager became agressive when I said the c...
1,France,The Frog at Bercy Village,Negative,A big disappointment,Feb 2024 •,"I ordered a beef fillet ask to be done medium,..."
2,France,The Frog at Bercy Village,Negative,Pretty Place with Bland Food,Nov 2023 •,"This is an attractive venue with welcoming, al..."
3,France,The Frog at Bercy Village,Negative,Great service and wine but inedible food,Mar 2023 •,Sadly I used the high TripAdvisor rating too ...
4,France,The Frog at Bercy Village,Negative,Avoid- Worst meal in Rome - possibly ever,Nov 2022 •,From the start this meal was bad- especially g...
...,...,...,...,...,...,...
1497,Cuba,Old Square (Plaza Vieja),Negative,The Tourism Trap,Oct 2016 •,Despite the other reviews saying that this is ...
1498,Cuba,Old Square (Plaza Vieja),Negative,the beer factory,Oct 2016 •,beer is good. food is awfull The only decent...
1499,Cuba,Old Square (Plaza Vieja),Negative,brewery,Oct 2016 •,"for terrible service of a truly comedic level,..."
1500,Cuba,Old Square (Plaza Vieja),Negative,It's nothing exciting over there,Oct 2016 •,We visited the Havana's Club Museum which is l...


In [2]:
import ray
import os

ray.init(address=f"ray://ray-head:10001",
        runtime_env={"pip": ["s3fs","numpy"]})

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2024-07-21 15:20:48,856	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Python version:,3.10.14
Ray version:,2.31.0
Dashboard:,http://172.20.0.4:8265


(ray_data_task pid=657) Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-07-21_08-08-44_678486_1/logs/ray-data
(ray_data_task pid=657) Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV]


(ray_data_task pid=657) Column           Type
(ray_data_task pid=657) ------           ----
(ray_data_task pid=657) Country          string
(ray_data_task pid=657) Restaurant Name  string
(ray_data_task pid=657) Sentiment        string
(ray_data_task pid=657) Review Title     string
(ray_data_task pid=657) Review Date      string
(ray_data_task pid=657) Review           string


(ray_data_task pid=657) Starting execution of Dataset. Full logs are in /tmp/ray/session_2024-07-21_08-08-44_678486_1/logs/ray-data
(ray_data_task pid=657) Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadCSV]


In [3]:
# check nodes
ray.nodes()

[{'NodeID': '85926493c9d1a54ad1f6c121b68a0cd462c51036dca8238dcf414be8',
  'Alive': True,
  'NodeManagerAddress': '172.20.0.6',
  'NodeManagerHostname': 'f569f4c5e6cb',
  'NodeManagerPort': 45651,
  'ObjectManagerPort': 32787,
  'ObjectStoreSocketName': '/tmp/ray/session_2024-07-21_08-08-44_678486_1/sockets/plasma_store',
  'RayletSocketName': '/tmp/ray/session_2024-07-21_08-08-44_678486_1/sockets/raylet',
  'MetricsExportPort': 60243,
  'NodeName': '172.20.0.6',
  'RuntimeEnvAgentPort': 46410,
  'DeathReason': 0,
  'DeathReasonMessage': '',
  'alive': True,
  'Resources': {'node:172.20.0.6': 1.0,
   'CPU': 1.0,
   'memory': 1409615872.0,
   'object_store_memory': 604121088.0},
  'Labels': {'ray.io/node_id': '85926493c9d1a54ad1f6c121b68a0cd462c51036dca8238dcf414be8'}},
 {'NodeID': 'b959acf69e86edf4500228556e862552415b7c9f4b1fea7925de6200',
  'Alive': True,
  'NodeManagerAddress': '172.20.0.5',
  'NodeManagerHostname': '61e93aa52f9b',
  'NodeManagerPort': 40235,
  'ObjectManagerPort': 33

In [4]:
#from pyarrow.fs import S3FileSystem
from s3fs.core import S3FileSystem
import os

In [5]:
fs =S3FileSystem(key=os.getenv('MINIO_ROOT_USER'), 
                 secret=os.getenv('MINIO_ROOT_PASSWORD'),
                 endpoint_url='http://storage:9000',
                 use_ssl=False)

In [8]:
ds = ray.data.read_csv("s3://default/European Restaurant Reviews.csv", filesystem=fs)

In [12]:
@ray.remote
def ray_data_task():
    return ds.schema() 
    #p1 = pd.DataFrame({'a': [3,4] * 10000, 'b': [5,6] * 10000})
    #ds = ray.data.from_pandas(p1)
    #return ds.repartition(4).to_pandas()

ray.get(ray_data_task.remote())

Column           Type
------           ----
Country          string
Restaurant Name  string
Sentiment        string
Review Title     string
Review Date      string
Review           string

In [8]:
!pip install numpy --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 5.2 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.0
    Uninstalling numpy-2.0.0:
      Successfully uninstalled numpy-2.0.0

[notice] A new release of pip is available: 23.0.1 -> 24.1.2
[notice] To update, run: pip install --upgrade pip
